# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Use the `@id` references to inspect the structure and contents.

In [ ]:
print("Available record sets and their fields:")
record_set_ids = [rset['@id'] for rset in metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    print('No record sets defined in the schema at top-level. Inspecting distribution or file objects for tabular data.')
    # Check distributions for tabular data
    for dist in metadata.to_json().get('distribution', []):
        print(json.dumps(dist, indent=2))
    # If mlcroissant finds records, let's print one as an example.
else:
    for rs_id in record_set_ids:
        print(f"Record set @id: {rs_id}")
        record_set = next((rs for rs in metadata.to_json().get('recordSet', []) if rs['@id']==rs_id), None)
        if record_set:
            field_ids = [f['@id'] for f in record_set.get('field', [])]
            print(f"  fields: {field_ids}")
        print()

# Try to print a few records
try:
    sample_records = list(dataset.records(record_set=None))
    print(f"Example record: {sample_records[0] if sample_records else 'No records loaded.'}")
except Exception as e:
    print(f"Error loading records: {e}")


## 3. Data Extraction
Load data from the primary record set into a DataFrame for analysis.

Since there are no explicit record sets in the Croissant schema, but the main tabular data is typically under the distribution, let's try to load all records without specifying a record set (or specifying `None`).

In [ ]:
# Extract all records from the default dataset (since record sets are not defined by @id explicitly)
records = list(dataset.records(record_set=None))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records.")
print("Columns (by field @id):\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering records based on criteria, normalizing numeric fields, and grouping data.

**NOTE:** All columns are referenced by their `@id` as loaded from the dataset. Let's inspect the numeric columns first.

In [ ]:
# Display all columns, select a numeric field for analysis
display_cols = df.columns.tolist()
print("First 10 columns:", display_cols[:10])
print("\nFirst data row:\n", df.iloc[0])
# Guess at a numeric field (by typical names)
possible_numeric_fields = [col for col in df.columns if 'Coverage' in col or 'MW' in col or 'Abundance' in col or col.lower().startswith('cr:')]
numeric_field = None
for col in possible_numeric_fields:
    try:
        _ = pd.to_numeric(df[col].iloc[0])
        numeric_field = col
        break
    except Exception:
        continue
if not numeric_field:
    print('No obvious numeric field found, inspecting all columns to find one...')
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col].iloc[0])
            numeric_field = col
            break
        except Exception:
            continue
if not numeric_field:
    raise ValueError('No numeric field found for analysis!')
print(f"Using numeric field for analysis (by @id): {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f}: {len(filtered_df)} rows")

# Normalize selected numeric field
filtered_df[f"{numeric_field}_normalized"] = filtered_df[numeric_field].astype(float).sub(filtered_df[numeric_field].astype(float).mean()).div(filtered_df[numeric_field].astype(float).std())
print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a field (e.g., description or protein type or similar)
possible_group_fields = [col for col in df.columns if 'Description' in col or 'Type' in col or 'Protein' in col or 'Group' in col]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
    print(f"Grouped filtered data by {group_field} and average {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot a histogram for the selected numeric field, and a bar plot for grouped averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].astype(float).dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Bar plot for top 10 groups if grouped_df exists
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10,4))
    top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    sns.barplot(data=top_groups, x=group_field, y=numeric_field, palette='viridis')
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored and processed the FAIR<sup>2</sup> dataset using the `mlcroissant` library. We loaded the metadata, examined the available fields (referenced by their `@id`s), extracted tabular data, performed basic EDA such as filtering and normalization, and visualized distributions.

All analyses referenced record fields via their schema `@id` for clarity and reproducibility. This approach ensures that future dataset schema changes (such as variable renaming) can be tracked precisely using persistent identifiers.

For downstream analysis, you can further tailor the notebook sections by domain knowledge, exploring relationships among the dataset's protein features, experimental outcomes, or annotations.